# ST-02 — Construction du périmètre siretisation

À partir de la sortie ST-01 (Phase 1), on récupère les **SIRET validés en P1** pour identifier les EG **non validés** qui passeront en Phase 2/3.

Pas de découpage A/B/C ici (la siretisation est plus directe que la sirenisation) : on prend tous les EG dont l'`idstructure_stru` n'apparaît pas dans Valide_fort/Valide de la P1.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.display      import afficher_tableau
from config.settings  import (
    FINESS_EG_CLEAN, ST_PHASE1, ST_PERIMETRE, PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Récupération des EG validés en Phase 1

In [2]:
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}

sheets_p1 = pd.read_excel(ST_PHASE1, sheet_name=None, dtype=str)
ids_valides = set()
sirets_valides = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES:
        if 'idstructure_stru' in sdf.columns:
            ids_valides.update(sdf['idstructure_stru'].dropna().astype(str).str.strip())
        if 'nmsiret_stru' in sdf.columns:
            sirets_valides.update(
                sdf['nmsiret_stru'].dropna().astype(str)
                .str.replace(r'\s', '', regex=True).str.strip()
            )
ids_valides.discard('')
sirets_valides.discard('')
print(f'EG validés en Phase 1 (idstructure)  : {len(ids_valides):,}')
print(f'SIRET validés en Phase 1 (uniques)   : {len(sirets_valides):,}')

EG validés en Phase 1 (idstructure)  : 60,258
SIRET validés en Phase 1 (uniques)   : 57,471


## 2. Construction du périmètre

In [3]:
df_eg = pd.read_parquet(FINESS_EG_CLEAN)
df_eg['_id_str'] = df_eg['idstructure_stru'].astype(str).str.strip()
print(f'EG FINESS total : {len(df_eg):,}')

df_perimetre = df_eg[~df_eg['_id_str'].isin(ids_valides)].copy()
df_perimetre = df_perimetre.drop(columns=['_id_str']).reset_index(drop=True)

print(f'\nEG à traiter        : {len(df_perimetre):,}')
print(f'EG exclus (validés en P1) : {len(df_eg) - len(df_perimetre):,}')

EG FINESS total : 104,805

EG à traiter        : 44,547
EG exclus (validés en P1) : 60,258


## 3. Aperçu

In [4]:
afficher_tableau(
    df_perimetre[['idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru',
                  'nmsiret_stru', 'raisonsociale_stru', 'cdcommune_stru']],
    f'Aperçu du périmètre siretisation ({len(df_perimetre):,} lignes)',
)

idstructure_stru,nmfinessej_stru,nmfinessetab_stru,nmsiret_stru,raisonsociale_stru,cdcommune_stru
2418254,070007950,070007968,44059019800014,EML CENTRE IMAGERIE MEDICALE TOURNON,07324
2418259,090004284,090004292,81433004900028,SAA BIENFAITS SERVICES,09261
2418262,090004375,090004383,48379417800044,SAAD GENERATIONS DOMICILE,09261
2418297,180009938,180009946,75257119000012,AIDOM SERVICES,18197
2418298,180009953,180009961,41442294900027,SAAD ASEF ST AMAND MONTROND,18197


## 4. Sauvegarde

In [5]:
df_perimetre.to_parquet(ST_PERIMETRE, index=False)
print(f'Sauvegardé : {ST_PERIMETRE}')

Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/processed/siretisation_perimetre.parquet
